In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [5]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/"
DATASET_ID = "produccion"
TABLE_ID= "ERRORES_desgravamen_TC_Periodos"
FECHA_PERIODO= "2025-11-10"


# SCRIPT COMPLETO

In [6]:
### CLIENTES DE STORAGE Y BIGQUERY
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

schema_desgravamen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR_SAS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_MOVIMIENTO_NUM", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        ]

#### FUNCION PARA LIMPIAR Y TRANSFORMAR LA COLUMNA A UN FORMATO DE FECHA
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


### LISTAR LOS ARCHIVOS QUE ESTAN DENTRO DEL BUCKET
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

### CICLO POR LA LISTA DE ARCHIVOS EXCEL
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    print(f" ------- CARGANDO ARCHIVO: {blob.name}")
    file = blob.download_as_string()
    df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Sheet1', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                                           'IDELOTE': str, 'CODIGO ERROR': str,
                                                                                           'SUMA ASEGURADA':str, 'TASA': str,
                                                                                           'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})
    ###### LIMPIEZA Y TRANSFORMACIONES

    # Colocar _ en los espacio de los nombres de las columnas
    df_desgravamen.columns = (df_desgravamen.columns.str.strip()
                                                    .str.upper()  # opcional: todo en mayúsculas
                                                    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                              )
    # Borrar Columnas repetidas
    df_desgravamen.drop(['IDELOTE','IDEDET_1'], axis=1, inplace=True)
    # Renombrar la columna
    df_desgravamen= df_desgravamen.rename(columns={'DESCRIPCION_ERROR':'DESCRIPCION_ERROR_SAS', 'CODIGO_ERROR':'IDEERROR'})

    # Imputar valores nulos
    df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].replace(r"^\s*$", None, regex=True)
    df_desgravamen["MONEDA"] = df_desgravamen["MONEDA"].where(df_desgravamen["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")
    df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
    df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'
    # Cambiar el sepador de decimales
    df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
    df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
    df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

    #Cambiar el tipo de datos de las columnas
    df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
    df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
    df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
    df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
    df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
    df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
    df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
    df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
    df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
    df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
    df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
    df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
    df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
    df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
    df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
    df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
    df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

    df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
    df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)
    df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
    # Columna para diferenciar el periodo de reporte de errores
    df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
    # Crear columna evaluando condiciones en el contenido de otras columnas
    df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)

    #Extraer datos de la la Trama Original
    df_desgravamen["TIPO_SEGURO"] = df_desgravamen["LINEA_TRAMA"].str[:3]
    df_desgravamen["TIPO_MOVIMIENTO_NUM"] = df_desgravamen["LINEA_TRAMA"].str[48]
    df_desgravamen["TIPO_REGISTRO"] = df_desgravamen["LINEA_TRAMA"].str[43:45]

    # Guardar tabla en BigQuery
    Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgravamen)
    print(f"### EL ARCHIVO: {blob.name} SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###")



 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/375 - errores_TC - 01.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/375 - errores_TC - 01.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/410 - errores_TC - 02.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/410 - errores_TC - 02.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/410 - errores_TC - 03.xlsx
### EL ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/410 - errores_TC - 03.xlsx SE HA GUARDADO CORRECTAMENTE EN BIGQUERY ###
 ------- CARGANDO ARCHIVO: data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/10-noviembre-2025/410 - errores_TC - 04.xlsx
### EL ARCHIVO: data_entries/REPORTES

--------------

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blob_errores = bucket.blob('data_entries/REPORTES DE ERRORES/tabla_errores2.xlsx')

In [ ]:
df_errores= pd.read_excel(BytesIO(blob_errores.download_as_string()), sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
#df_errores.drop(['IDEERROR'], axis=1, inplace=True)
df_errores.head(3)

,IDEERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,0974,1068,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,CONFIGURACION SAS,ANULAR MASIVAS
1,0005,0003,ERROR EN CARGA DE TRAMA,ERROR EN CARGA DE TRAMA,ANALISIS EMISOR,NaN
2,1158,1155,TRAMA REPETIDA O DUPLICADA,TRAMA REPETIDA O DUPLICADA,CONFIGURACION SAS,ANULAR MASIVAS


In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blobs_excels = list(bucket.list_blobs(prefix=FOLDER_PATH))

In [ ]:
for blob in blobs_excels:
  if blob.name.endswith(".xlsx"):
    #file = blob.download_as_string()
    #excel_file = pd.ExcelFile(file)
    print(blob.name)
    #print(excel_file.sheet_names)

data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 01.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 02.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 03.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 04.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 05.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 06.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 07.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 08.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 09.xlsx
data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 10.xlsx
data_entries/REPORTES DE ERROR

In [ ]:
#blob_consuer = bucket.blob('desgravamen_prestamos/desgravamen - consuer.csv')
#file = blob_consuer.download_as_string()
#df_desgravamen= pd.read_csv(BytesIO(file), dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 'IDELOTE': str, 'CODIGO ERROR': str})

In [ ]:
print(blobs_excels[1].name)
file = blobs_excels[1].download_as_string()
df_desgravamen= pd.read_excel(BytesIO(file), sheet_name='Sheet1', dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str,
                                                                                           'IDELOTE': str, 'CODIGO ERROR': str,
                                                                                           'SUMA ASEGURADA':str, 'TASA': str,
                                                                                           'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})


data_entries/REPORTES DE ERRORES/DESGRAVAMEN/2. TC/18-septiembre-2025/410 - errores_TC - 01.xlsx


In [ ]:
df_desgravamen.head(3)

,NRO LOTE,LOTES ANTERIORES,FECHA CARGA,CODIGO PRODUCTO,PRODUCTO,CODIGO PLAN,NOMBRE DE PLAN,COD DE CERTIFICADO,fecini_alta_CERTIFICADO,fecfin_alta_CERTIFICADO,IDEDET,TIPO MOVIMIENTO,FEC. INICIO,FEC. FIN,MONEDA,SUMA ASEGURADA,TASA,TASA RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE DE ARCHIVO,LINEA TRAMA,IDELOTE,IDEDET.1,ORIGEN ERROR,CODIGO ERROR,DESCRIPCION ERROR
0,40856,NaN,2015-03-16,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110125254000126298,NaT,NaT,65909536,Renovacion,19/09/2014,09/01/2015,SOL,4.24,0,0,5,4.85,PAOLA,PEREZ,PINEDO,19830616,2,41961269,20100130204_0158001_20150310_002.TXT,918001101252540001262980011012521500140148501P...,40856,65909536.0,Error canal,1153,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00110125...
1,41817,NaN,2015-04-01,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110133684000194238,NaT,NaT,69920930,Exclusion,NaN,24/02/2015,SOL,0,0,0,0,0,ERIKA SHEILA,MENESES,CUZCANO,19830709,2,41957497,20100130204_0158001_20150331_006.TXT,918001101336840001942380011013360500143789001P...,41817,69920930.0,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011013368...
2,41817,NaN,2015-04-01,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110785334000394305,NaT,NaT,69921506,Exclusion,NaN,19/02/2015,SOL,0,0,0,0,0,MIGUEL ANGEL,ARRIARAN,ESCRIBA,19830309,2,43723290,20100130204_0158001_20150331_006.TXT,918001107853340003943050011078533500198853101P...,41817,69921506.0,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011078533...


In [ ]:
df_desgravamen.columns = (
    df_desgravamen.columns
    .str.strip()  # quitar espacios al inicio/fin
    .str.upper()  # opcional: todo en mayúsculas
    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)
df_desgravamen.drop(['IDELOTE','IDEDET_1'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.rename(columns={'CODIGO_ERROR': 'IDEERROR', 'DESCRIPCION_ERROR':'DESCRIPCION_ERROR_SAS'})
#df_desgravamen.drop(['IDEDET_1','ORIGEN_ERROR','UNNAMED__32'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.rename(columns={'IDELOTE': 'LINEA_TRAMA','CODIGO_ERROR':'ORIGEN_ERROR', 'DESCRIPCION_ERROR': 'CODIGO_ERROR'})

In [ ]:
df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'MONEDA': 'SIN DATO', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_desgravamen.loc[df_desgravamen['MONEDA'] == 'nan', 'MONEDA'] = 'SIN DATO'
df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

In [ ]:
df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [ ]:
df_desgravamen.head()

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,DESCRIPCION_ERROR_SAS
0,40856,,2015-03-16,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110125254000126298,NaT,NaT,65909536,Renovacion,19/09/2014,09/01/2015,SOL,4.24,0,0,5,4.85,PAOLA,PEREZ,PINEDO,19830616,2,41961269,20100130204_0158001_20150310_002.TXT,918001101252540001262980011012521500140148501P...,Error canal,1153,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00110125...
1,41817,,2015-04-01,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110133684000194238,NaT,NaT,69920930,Exclusion,NaN,24/02/2015,SOL,0,0,0,0,0,ERIKA SHEILA,MENESES,CUZCANO,19830709,2,41957497,20100130204_0158001_20150331_006.TXT,918001101336840001942380011013360500143789001P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011013368...
2,41817,,2015-04-01,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110785334000394305,NaT,NaT,69921506,Exclusion,NaN,19/02/2015,SOL,0,0,0,0,0,MIGUEL ANGEL,ARRIARAN,ESCRIBA,19830309,2,43723290,20100130204_0158001_20150331_006.TXT,918001107853340003943050011078533500198853101P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011078533...
3,41817,,2015-04-01,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110199754000532419,NaT,NaT,69921059,Exclusion,NaN,11/02/2015,SOL,0,0,0,0,0,JUAN ANTONIO,RAMOS,GOMEZ,19741005,2,10263073,20100130204_0158001_20150331_006.TXT,918001101997540005324190011019977500146813301P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011019975...
4,41817,,2015-04-01,4046.0,Desgravamen BBVA - T. Credito + Desempleo,86526.0,Plan Saldo insoluto soles,00110301924000910351,NaT,NaT,69921267,Exclusion,NaN,26/02/2015,SOL,0,0,0,0,0,CYNTHIA PATRICIA,MONCADA,CHOTA,19850713,2,43065903,20100130204_0158001_20150331_006.TXT,918001103019240009103510011030192500217979001P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011030192...


In [ ]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [ ]:
df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
#df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce")
#df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce")
#df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce")
#df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce")
#df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce")
df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")
df_desgravamen['DESCRIPCION_ERROR_SAS'] = df_desgravamen['DESCRIPCION_ERROR_SAS'].astype(str)
#df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else np.nan)
#df_desgravamen['CODIGO_ERROR'] = df_desgravamen['CODIGO_ERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [ ]:
df_desgravamen['CODIGO_PRODUCTO'].value_counts()

,count
CODIGO_PRODUCTO,
4046,87347
0,255


In [ ]:
#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [ ]:
df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<30000]   #Eliminar las filas con valores atipicos muy altos
df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)    # Crear colunma evaluando condiciones en el contenido de otras columnas

In [ ]:
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,DESCRIPCION_ERROR_SAS,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR
0,40856,,2015-03-16,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110125254000126298,NaT,NaT,65909536,Renovacion,2014-09-19,2015-01-09,SOL,4.24,0.0,0.0,5.0,4.85,PAOLA,PEREZ,PINEDO,1983-06-16,2,41961269,20100130204_0158001_20150310_002.TXT,918001101252540001262980011012521500140148501P...,Error canal,1153,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00110125...,2015-03-10,2025-09-05,Mes corriente
1,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110133684000194238,NaT,NaT,69920930,Exclusion,NaT,2015-02-24,SOL,0.00,0.0,0.0,0.0,0.00,ERIKA SHEILA,MENESES,CUZCANO,1983-07-09,2,41957497,20100130204_0158001_20150331_006.TXT,918001101336840001942380011013360500143789001P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011013368...,2015-03-31,2025-09-05,Mes corriente
2,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110785334000394305,NaT,NaT,69921506,Exclusion,NaT,2015-02-19,SOL,0.00,0.0,0.0,0.0,0.00,MIGUEL ANGEL,ARRIARAN,ESCRIBA,1983-03-09,2,43723290,20100130204_0158001_20150331_006.TXT,918001107853340003943050011078533500198853101P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011078533...,2015-03-31,2025-09-05,Mes corriente


In [ ]:
df_desgravamen[df_desgravamen['IDEERROR']== ''].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,DESCRIPCION_ERROR_SAS,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR
46,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110102184000211087,2015-01-20,2067-03-11,69920849,Exclusion,NaT,2015-02-24,SOL,0.0,0.0,0.0,4.5,4.37,CARLOS JAVIER,ARCOS,ANDIA,1987-03-12,2,44121077,20100130204_0158001_20150331_006.TXT,918001101021840002110870011010219500158923301P...,SIN DATO,,nan,2015-03-31,2025-09-05,Mes corriente
47,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110110404000102389,2015-01-20,2023-07-15,69920865,Exclusion,NaT,2015-02-24,SOL,0.0,0.0,0.0,3.0,2.91,CARMELO,MAYORGA,VALDEIGLESIAS,1943-07-16,2,00062825,20100130204_0158001_20150331_006.TXT,918001101104040001023890011011045500123410201P...,SIN DATO,,nan,2015-03-31,2025-09-05,Mes corriente
48,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110115354000268853,2015-01-20,2031-06-23,69920881,Exclusion,NaT,2015-02-24,SOL,0.0,0.0,0.0,4.5,4.37,ELVIRA JUANA,EGUSQUIZA,ELGUERA,1951-06-24,2,07306030,20100130204_0158001_20150331_006.TXT,918001101153540002688530011083154500171648401P...,SIN DATO,,nan,2015-03-31,2025-09-05,Mes corriente


In [ ]:
df_desgravamen['TIPDOCUMENTO'].value_counts()

,count
TIPDOCUMENTO,
2,78785
1,8128
4,619
6,40
SIN DATO,14
S,2
0,2


In [ ]:
df_desgravamen[df_desgravamen['TIPDOCUMENTO'].isnull()].head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,CODIGO_ERROR,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR


In [ ]:
df_desgravamen= df_desgravamen.merge(df_errores, on='IDEERROR', how='left')
#df_desgravamen.drop(['IDEERROR'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,IDEERROR,DESCRIPCION_ERROR_SAS,FECHA_TRAMA,FECHA_PERIODO,TIPO_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,40856,,2015-03-16,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110125254000126298,NaT,NaT,65909536,Renovacion,2014-09-19,2015-01-09,SOL,4.24,0.0,0.0,5.0,4.85,PAOLA,PEREZ,PINEDO,1983-06-16,2,41961269,20100130204_0158001_20150310_002.TXT,918001101252540001262980011012521500140148501P...,Error canal,1153,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00110125...,2015-03-10,2025-09-05,Mes corriente,1150,ERROR EN ALTA,EL ALTA DEL NRO. CERTIFICADO EXTERNO SE ENCUEN...,ANALISIS EMISOR,NaN
1,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110133684000194238,NaT,NaT,69920930,Exclusion,NaT,2015-02-24,SOL,0.00,0.0,0.0,0.0,0.00,ERIKA SHEILA,MENESES,CUZCANO,1983-07-09,2,41957497,20100130204_0158001_20150331_006.TXT,918001101336840001942380011013360500143789001P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011013368...,2015-03-31,2025-09-05,Mes corriente,1152,ERROR EN ALTA,NO EXISTE ALTA PARA EL CERTIFICADO O TIENE BAJ...,ANALISIS EMISOR,NaN
2,41817,,2015-04-01,4046,Desgravamen BBVA - T. Credito + Desempleo,86526,Plan Saldo insoluto soles,00110785334000394305,NaT,NaT,69921506,Exclusion,NaT,2015-02-19,SOL,0.00,0.0,0.0,0.0,0.00,MIGUEL ANGEL,ARRIARAN,ESCRIBA,1983-03-09,2,43723290,20100130204_0158001_20150331_006.TXT,918001107853340003943050011078533500198853101P...,Error canal,1150,NO EXISTE ALTA PARA EL CERTIFICADO: 0011078533...,2015-03-31,2025-09-05,Mes corriente,1152,ERROR EN ALTA,NO EXISTE ALTA PARA EL CERTIFICADO O TIENE BAJ...,ANALISIS EMISOR,NaN


In [ ]:
df_desgravamen['DESCRIPCION_ERROR'].value_counts()

,count
DESCRIPCION_ERROR,
ERROR EN ALTA,40812
NI ÉXITO NI ERROR,6134
CAMPOS EN BLANCO,315
PAGO PENDIENTE DE CORREGIR,62
TRAMA REPETIDA O DUPLICADA,20
VALIDACIONES CONSECUENCIA ACSEL E,5
ERROR VIGENCIA,4


In [ ]:
df_desgravamen["DESCRIPCION_ERROR"] = df_desgravamen["DESCRIPCION_ERROR"].astype(str)
df_desgravamen["DETALLE"] = df_desgravamen["DETALLE"].astype(str)
df_desgravamen["SOLUCION"] = df_desgravamen["SOLUCION"].astype(str)
df_desgravamen["SOLUCION_2"] = df_desgravamen["SOLUCION_2"].astype(str)

In [ ]:
df_desgravamen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87590 entries, 0 to 87589
Data columns (total 39 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   NRO_LOTE                 87590 non-null  int64         
 1   LOTES_ANTERIORES         87590 non-null  object        
 2   FECHA_CARGA              87590 non-null  datetime64[ns]
 3   CODIGO_PRODUCTO          87590 non-null  int64         
 4   PRODUCTO                 87590 non-null  object        
 5   CODIGO_PLAN              87590 non-null  int64         
 6   NOMBRE_DE_PLAN           87590 non-null  object        
 7   COD_DE_CERTIFICADO       87590 non-null  object        
 8   FECINI_ALTA_CERTIFICADO  47259 non-null  datetime64[ns]
 9   FECFIN_ALTA_CERTIFICADO  47259 non-null  datetime64[ns]
 10  IDEDET                   87590 non-null  int64         
 11  TIPO_MOVIMIENTO          87590 non-null  object        
 12  FEC__INICIO              210 non

In [ ]:
TABLE_ID= "ERRORES_desgravamen_TC_TEST"
schema_desgramen = [
        bigquery.SchemaField("NRO_LOTE", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("LOTES_ANTERIORES", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PLAN", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("NOMBRE_DE_PLAN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("COD_DE_CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECINI_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECFIN_ALTA_CERTIFICADO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.INTEGER),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN ", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SUMA_ASEGURADA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("TASA_RECARGO", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMABRUTACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("PRIMANETACAN", bigquery.enums.SqlTypeNames.FLOAT),
        bigquery.SchemaField("NOMCOMPLETO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEPATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("APEMATERNO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECNACIMIENTO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMDOCUMENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("ORIGEN_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR_SAS", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_PERIODO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("TIPO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DESCRIPCION_ERROR", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("DETALLE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("SOLUCION_2", bigquery.enums.SqlTypeNames.STRING)
    ]
Guardar_en_BigQuery(df_desgravamen, DATASET_ID, TABLE_ID, schema_desgramen)

----- Se ha creado la tabla ERRORES_desgravamen_TC_TEST en el dataset produccion -----
----- REGISTROS AGREGADOS CORRECTAMENTE EN: ERRORES_desgravamen_TC_TEST -------
